[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flatironinstitute/2026-flatiron-cmb-summer-school/blob/main/notebooks/Intro_to_FFTs.ipynb)

In this notebook, we will give an overview of the DFT -- [the discrete Fourier transform](https://en.wikipedia.org/wiki/Discrete_Fourier_transform). This mathematical operation is central to all aspects of cosmology analysis (and any kind of signal processing), so having a baseline to lean on will be very useful as you move through the data school.

Let's jump right in!

# The 1-dimensional DFT

Put simply, the DFT *decomposes* or *expands* a discrete (digital) signal, say $f(x_i)$ into its *frequency components*. Because we will be dealing with digital data throughout the school (and in general in cosmology), we can think of $x_i$ is the position along the x-axis in *discrete* steps. The simplest example of this is a 1d numpy array:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f = np.random.random(10)
x = np.arange(len(f))

plt.plot(x, f, marker='o')
plt.xticks(x)
plt.xlabel('x')
plt.ylabel('f(x)')
plt.show()

Notice how the data are only defined at discrete values of the independent variable, $x$, where the blue dots are. The "line" between them is just a guide to the eye that interpolates between the data points. Of course, with enough data points, we can represent smooth functions to very high resolution, for example this "damped" oscillator:

In [ ]:
decay_time = 2.5 # characteristic time for oscillator to decay by 1/e on average
freq = 2.0  # frequency (cycles/second)
phase = 0.15 # starting position (fraction of first cycle)

t = np.linspace(0, 5, 1000) # 1000 points from 0 to 5 seconds

# calculate the damped oscillator
y = np.exp(-t/decay_time) * np.cos(2*np.pi*(freq * t + phase))

plt.plot(t, y)
plt.title('Damped Oscillator at High Resolution')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True)
plt.show()

That looks pretty smooth. Indeed, this is no different than the data on your screen, or coursing through your phone when you open an app --- we live in a digital world. When we make/simulate/analyze maps of the microwave sky in this school, they too will be *digital*, with their data defined in discrete *pixels*.

Why the emphasis on discrete data? Let's hold onto that question for a second. First, we have to motivate and define the DFT.

Imagine you're driving in your car and decide to play a genuine banger (source [here](https://commons.wikimedia.org/wiki/File:Twinkle_Twinkle_Little_Star_plain.ogg)):

In [ ]:
import scipy.io.wavfile as wavfile

!wget https://raw.githubusercontent.com/flatironinstitute/2026-flatiron-cmb-summer-school/main/notebooks/Twinkle_Twinkle_Little_Star_plain.wav

wav_file = 'Twinkle_Twinkle_Little_Star_plain.wav'

# load the wav file into a numpy array
samplerate, data = wavfile.read(wav_file)

# we'll only keep the first 4 seconds
data = data[:samplerate*4]

print(f"sample rate: {samplerate} Hz")
print(f"audio data shape: {data.shape}")

# play the banger. this function assumes data with shape (channels, waveforms),
# but it's loaded like (waveforms, channels) so we need the transpose
from IPython.display import Audio

Audio(data=data.T, rate=samplerate)

This data represents the actual signal emitted by your speakers (there are two channels, one for each stereo speaker) as a function of time. It's pretty finely sampled --- 44,100 times per second --- but it's still *discrete*. We can plot the whole signal, and a zoomed-in portion to understand the data:

In [ ]:
t = np.linspace(0, len(data)/samplerate, len(data), endpoint=False) # time in seconds

plt.plot(t, data[:, 0], alpha=0.5, label='left speaker')
plt.plot(t, data[:, 1], alpha=0.5, label='right speaker')
plt.legend()

plt.title('full timestream of banger')
plt.xlabel('time (s)')
plt.ylabel('amplitude')
plt.show()

plt.title('zoomed-in timestream of banger')
mask = np.logical_and(t > 1, t < 1.02)
plt.plot(t[mask], data[mask, 0], alpha=0.5, label='left speaker')
plt.plot(t[mask], data[mask, 1], alpha=0.5, label='right speaker')
plt.legend()
plt.xlabel('time (s)')
plt.ylabel('amplitude')
plt.show()

Clearly there is some oscillatory behavior, but it's kind of complicated. Added up and played over your speaker, your ear does a good job at interpreting this signal.

Now, your car's audio system likely has some settings called "bass," "mid," and "treble." By adjusting these, you can raise or lower the volume of the deepest pitches, the mid-range pitches, and the highest pitches, respectively. How does this work? We need some way of measuring the "amount" of each frequency, adjusting these amounts, and then putting the adjusted signal back together. This is exactly what the DFT (and its inverse) provides. Since we'll be using numpy to perform the transforms, we'll stick to [its conventions](https://numpy.org/doc/stable/reference/routines.fft.html#implementation-details):

![numpy_fft.png](https://raw.githubusercontent.com/flatironinstitute/2026-flatiron-cmb-summer-school/main/notebooks/numpy_fft.png)

Here, $a_m$ is what we have been calling our data, and $m$ is the index of the signal. There are $n$ such indexes --- the length of the numpy array --- and they are all summed over. The output is a new array, also of length $n$, $A_k$, which is the *convolution* of the signal with the complex exponential evaluated at each index $m$ and $k$. This output, $A_k$ *is* exactly what we're looking for, the "amount" of frequency $f=k/n$ that makes up $a_m$. This is the DFT!

Notice a few features:
* $n$ discrete inputs ($a_m$) gives $n$ discrete outputs ($A_k$)
* $A$ is complex: it has both real and imaginary parts
* $A_k$ is insensitive to a permutation of either $m$ or $k$ by $n$ (e.g., $k→k+n$)

Before trying some examples, we need one more piece of prep-work: what are/how do we keep track of the frequencies? Numpy also does this for us with the [numpy.fft.fftfreq](https://numpy.org/doc/stable/reference/generated/numpy.fft.fftfreq.html) function, and in fact the screenshot above is a little deceptive. Let's plot it:

In [ ]:
# let's make a simple plot of the fftfreq outputs for a small number of inputs
N = 10

fftfreqs = np.fft.fftfreq(N)

plt.plot(fftfreqs, marker='o')
plt.xlabel('frequency index k')
plt.ylabel('frequency (k/n)')

Ok, there are 10 frequencies as expected, but their ordering is important to get used to:
* First, numpy doesn't use the range `k=0..n-1` as described, rather (since the DFT is insensitive to `n`-length shifts!), it uses `k=-n/2..n/2`. Remember, the *frequencies* here are $f=k/n$.
* Second, the order starts with `0`, then goes up to `n/2`, then immediately jumps down to `-n/2`, then starts going back up toward `0`. This is just a convention in how numpy works! There's nothing fundamentally problematic with this, we just have to remember this convention!
* Last, the maximum frequency is `n/2` (which is the same as `-n/2` for even-length sequences; numpy chooses to assign it to `-n/2` *by convention*). This is a *very important frequency*. It is called the [Nyquist frequency](https://en.wikipedia.org/wiki/Nyquist_frequency). Never forget it: "for an `n`-length sequence, the highest possible frequency in the sequence is `n/2`"

Don't be alarmed by these weird conventions! We just need to make sure we remember and abide by them.

Finally, `numpy.fft.fftfreq` allows you to assign units to the frequencies, say if you know that your signal data is sampled every `d` seconds, or meters, or degrees etc. By default `d=1`. In our case:


In [ ]:
# we have 44100 samples per second, so the sample spacing is one over that
d = 1/samplerate

fftfreqs = np.fft.fftfreq(len(data), d)

plt.plot(fftfreqs, marker='o')
plt.xlabel('frequency index k')
plt.ylabel('frequency (k/nd) [Hz or cycles/second]')

So the maximum frequency that can be sampled in our soundtrack is 22050 Hz!

Let's prove it to ourselves that the DFT does indeed measure the "amount" of signal at a single frequency. We'll construct a fake signal that has one frequency, and then look at the DFT that numpy provides:

In [ ]:
# pick a signal frequency and a phase
# !!! Try changing these !!!
freq = 1 # Hz
phase = 0.15 # fraction of a cycle of the tone

# let's downsample the timestream so it's easier to see plots.
# we'll do this by imagining that there are only 40 samples in the
# 4 seconds rather than the million are so in the actual data
N = 40
samplerate = N/4 # 40 samples in 4 seconds so 10 samples per second
d = 1/samplerate # sample spacing is 1/samplerate

t = np.linspace(0, 4, N, endpoint=False) # time in seconds
single_freq_signal = np.cos(2*np.pi*(freq * t + phase))

# first plot the tone
plt.plot(t, single_freq_signal)
plt.title('single frequency signal')
plt.xlabel('time (s)')
plt.ylabel('signal amplitude')
plt.show()

# now plot the fft of the tone as a function of its frequency!
# remember that there are *real* and *imaginary* parts since
# the fft is complex!
fftfreqs = np.fft.fftfreq(N, d)
fft_of_single_freq_signal = np.fft.fft(single_freq_signal)

plt.plot(fftfreqs, fft_of_single_freq_signal.real, marker='o', label='real')
plt.plot(fftfreqs, fft_of_single_freq_signal.imag, marker='o', label='complex')
plt.title('DFT of single frequency signal')
plt.xlabel('frequency (k/nd) [Hz or cycles per second]')
plt.ylabel('fft amplitude')
plt.legend()
plt.show()

Indeed, this seems to work! Well, sort of. If you keep the original settings (1 Hz frequency and a phase of 0.15 cycles), you'll see that the DFT is only non-zero at 1 Hz *and* -1 Hz, and it seems to have both real *and* complex components...

What's going on here?

Try changing the phase and describe to yourself what happens! Can you make the real part disappear? The imaginary part? Can you make the spike at -1 Hz disappear?

This brings us to two more important properties of the DFT (really, any Fourier transform). Try to convince yourself of these by playing with the previous plots, or by looking at the math of the DFT!
* The *complex phase* of $A_k$ encodes the *phase* of the single-frequency signal with frequency corresponding to $k$
* If the signal is *real* (i.e., no complex part), then $A_k$ obeys the symmetry relation $A_{-k} = A_k^*$

Another property is that the Fourier transform is *linear*: the DFT of signal $a_m$ plus the DFT of signal $b_m$ is the DFT of the signal $a_m + b_m$. Try adding *two* (or more) single-frequency waves and plotting their DFT. Is it what you expect?

In [ ]:
# Exercise: use the code in the previous cell (you could wrap it in a function!)
# to plot the DFT of the *sum* of two single-frequency waves. You could start
# with these settings. Does the plot of the DFT make sense?
freq1 = 1 # Hz
phase1 = 0.15 # fraction of a cycle

freq2 = 2 # Hz
phase2 = 0.25 # fraction of a cycle

# your code here

Ok, so it seems like the DFT really does measure the amplitude and phase of each frequency component in a given signal. But, is *any* discrete signal really composed of a sum of frequency components? The amazing thing is, yes! In comes the *inverse* DFT (again in [numpy's convention](https://numpy.org/doc/stable/reference/routines.fft.html#implementation-details)):

![numpy_ifft.png](https://raw.githubusercontent.com/flatironinstitute/2026-flatiron-cmb-summer-school/main/notebooks/numpy_ifft.png)

This looks basically like the DFT, except the sign of the exponential is flipped, the sum is over $k$ instead of $m$, and the $1/n$ in the front. Fundamentally though, the iDFT comes with the same features and principles as the DFT. Anyway, this tells us that *any* signal $a_m$ is indeed the sum of waves (the complex exponentials) multiplied by complex components $A_k$. In other words, given the Fourier transform of a signal $A_k$, we can recover the signal using the iDFT.

Let's confirm this is true by looking first at one frequency at a time:

In [ ]:
# start with the DFT of our single frequency from two cells before.

# same code to plot as before
plt.plot(fftfreqs, fft_of_single_freq_signal.real, marker='o', label='real')
plt.plot(fftfreqs, fft_of_single_freq_signal.imag, marker='o', label='complex')
plt.title('DFT of single frequency signal')
plt.xlabel('frequency (k/nd) [Hz or cycles per second]')
plt.ylabel('fft amplitude')
plt.legend()
plt.show()

# what signal does it correspond to?
signal_from_inverse_fft = np.fft.ifft(fft_of_single_freq_signal)
plt.plot(t, signal_from_inverse_fft)
plt.title('signal from inverse DFT')
plt.xlabel('time (s)')
plt.ylabel('signal amplitude')
plt.show()

We should have gotten our original single-frequency wave back (as expected)! If not, check to make sure you ran the cell that defines `fft_of_single_freq_signal` just before you run this, in case you were playing around with the plots a bit.

Careful though, that warning tells us that actually `signal_from_inverse_fft` is complex. If everything worked, it's just that its imaginary part is 0 (since we should have gotten the original single-frequency wave back, which we defined as real):

In [ ]:
# what signal does it correspond to?
signal_from_inverse_fft = np.fft.ifft(fft_of_single_freq_signal)
plt.plot(t, signal_from_inverse_fft.real, label='real part of signal')
plt.plot(t, signal_from_inverse_fft.imag, label='imaginary part of signal')
plt.title('signal from inverse DFT')
plt.xlabel('time (s)')
plt.ylabel('signal amplitude')
plt.legend()
plt.show()

Now try to do the inverse of your two-frequency case. Do you get back your original signal using the inverse DFT (`numpy.fft.ifft`)?

In [ ]:
# your code here

The warning about `Casting complex values to real discards the imaginary part` is telling us something interesting, but also that we need to be careful. What would happen to our signal in time if we took the `fft_of_single_freq_signal` array but set the negative-frequency part to 0?

In [ ]:
# what if we set the negative frequency components to 0?

# NOTE: don't really need to do anything special to handle that the fftfreqs
# are in an "unintuitive ordering"
keep_these_frequencies = fftfreqs >= 0 # nothing fancy or tricky here!
fft_with_positive_freqs_only = fft_of_single_freq_signal * keep_these_frequencies # the multiplication here sets the negative freqs to 0

# same code to plot fft as before
plt.plot(fftfreqs, fft_with_positive_freqs_only.real, marker='o', label='real')
plt.plot(fftfreqs, fft_with_positive_freqs_only.imag, marker='o', label='complex')
plt.title('DFT of single frequency signal')
plt.xlabel('frequency (k/nd) [Hz or cycles per second]')
plt.ylabel('fft amplitude')
plt.legend()
plt.show()

# same code to plot inverse fft as before
signal_from_inverse_fft = np.fft.ifft(fft_with_positive_freqs_only)
plt.plot(t, signal_from_inverse_fft.real, label='real part of signal')
plt.plot(t, signal_from_inverse_fft.imag, label='imaginary part of signal')
plt.title('signal from inverse DFT')
plt.xlabel('time (s)')
plt.ylabel('signal amplitude')
plt.legend()
plt.show()

We (intentionally) broke the symmetry relationship $A_{-k} = A_k^*$ necessary for the signal to be real, and so it wasn't anymore. In fact we picked out exactly one complex exponential from the inverse DFT sum (look at the equation for the inverse-DFT above and set *exactly one* $A_k$ to be nonzero).

So why do we need to be careful? If we want to build a signal $a_m$ in real space (time, distance, etc) from its Fourier-space components $A_k$ using something like `a_m = np.fft.ifft(A_k).real`, we will pick out from $A_k$ *only* the part that satisfies $A_{-k} = A_k^*$. Clearly, make sure that is what you intended! (If you *know* your input signal is *real*, you could also use "real" ffts, as implemented in [np.fft.rfft](https://numpy.org/doc/stable/reference/generated/numpy.fft.rfft.html), to avoid this issue.)

Actually, the fact that the inverse-DFT produces a complex signal (in general) is the key to why *any* signal can be represented as a sum of complex exponentials. In other words, let's prove that you can find a set of $A_k$ such that:
$$a_m = 1/n\sum_{k=0}^{n-1}A_k \exp(2\pi i m k /n)$$
is true. Well, we'll leave the proof for you to try if you like, but the answer is (perhaps not surprisingly):
$$A_k = \sum_{m=0}^{n-1}a_m\exp(-2\pi i m k /n)$$
Here's the clincher...this works equally well if we *exchange* the definitions of the DFT and inverse DFT. In other words, numpy *could* have chosen the *equally valid* definitions:
$$a_m = \sum_{k=0}^{n-1}A_k\exp(-2\pi i m k /n)$$
and
$$A_k = 1/n\sum_{m=0}^{n-1}a_m \exp(2\pi i m k /n)$$
where we have simply *swapped* Fourier space and real space. There is no difference between Fourier space and real space mathematically speaking --- you just need to be consistent: pick one of the DFT or the inverse-DFT to go from one "space" to the other "space", and pick the other transformation to go back!

An important consequence of this is the following "rule of thumb:" we've already seen that "functions that are *very local* (like, spikes) in Fourier space are *very non-local* (like, never-ending waves) in real space." But what is the Fourier transform of a spike in real space? Try it! What's the rest of the "rule of thumb"?

In [ ]:
# a "spike" in real space can be an empty array except for one nonzero value
spike_size = 10

array_length = 50
spike_position = 12

# spiky data
spiky_data = np.zeros(array_length)
spiky_data[spike_position] = spike_size

# plot it
plt.plot(spiky_data)
plt.show()

# what's the Fourier transform look like? remember it is complex, plot the real
# and imaginary parts


This is perhaps an abstract mathematical lesson that we don't need to worry about too much. We can stick to using the DFT to go from "real" space to "Fourier" space, and the inverse-DFT to go back. There will be another very important rule related to the above math lesson that we need to follow when it comes to convolutions; we'll get to that in a bit.

First, let's go back to your car radio and put your skills to work. Can you implement the "bass", "mid", and "treble" adjusters to modify the song snippet? What does the modified song sound like?

In [ ]:
# we'll outline the solution *a little*. you do the rest!

# let's grab the data from before
samplerate = 44100

t = np.linspace(0, len(data)/samplerate, len(data), endpoint=False) # time in seconds
print(data.shape) # to remember, or scroll up

# build out this function:
def adjust_bass_mid_treble(t, data, bass=1, mid=1, treble=1):
  """Adjust the lowest one-third frequencies by the relative
  amount passed in 'bass', the middle one-third of frequences
  by the relative amount passed in 'mid', and the highest
  one-third of frequencies by the relative amount passed in
  'treble'.

  Parameters
  ----------
  t : (n,) np.ndarray
    The time array in seconds. Has n elements.
  data : (n, 2) np.ndarray
    The data array. Has n rows of "stereo" (two-channel) data.


  """
  # sanity checks
  assert len(t) == len(data)

  # np.fft.fft and np.fft.ifft compute the transforms over the "last" axis in
  # the passed array by default. but the "time" axis is the first axis in data.
  # either transpose data or pass axis=0 to the fft functions

  # when adjusting the frequency components, remember we want to respect
  # A_{-k} = A_k^*! That way we can safely return only the real part of the
  # output signal, since we know the complex part was 0 anyway.
  #
  # make sure of this!
  assert np.allclose(adjusted_data.imag, 0)

  return adjusted_data.real

# adjust the song how you like. these settings should amplify the bass notes
bass = 1.2
mid = 0.8
treble = 0.8

adjusted_data = adjust_bass_mid_treble(t, data, bass=bass, mid=mid, treble=treble)

# you can plot the adjusted_data if you like

# or play it over audio!
Audio(data=adjusted_data.T, rate=samplerate)


# Convolutions

Why is it important to learn about (discrete) Fourier transforms for a practical cosmology course? There are probably three main reasons:
1. Fourier transforms are very useful for solving differential equations (like, the physical theories that govern cosmological observables) and taking derivatives. We'll encounter this during the course
2. Fourier transforms are very useful at *diagonalizing translationally-invariant correlation patterns*. We'll encounter this during the course
3. Fourier transforms are very useful for quickly implementing a convolution

A [convolution](https://en.wikipedia.org/wiki/Convolution) is like "smoothing" one function by another function (sometimes, the function doing the smoothing is called the "kernel"):

![convolution.png](https://raw.githubusercontent.com/flatironinstitute/2026-flatiron-cmb-summer-school/main/notebooks/convolution.png)

Here, $g$ is the function we'd like to smooth, and $f$ is the smoothing kernel.

A simple example might be smoothing a spiky function with a "double-cosine" profile:

In [ ]:
N = 200

# first let's get a spiky function that we want to smooth
spiky_points = [6, 52, 70, 186]
plateau_regions = [[100, 124], [142, 160]]

spiky_data = np.zeros(N)
for spiky_point in spiky_points:
  spiky_data[spiky_point] = 1
for plateau_region in plateau_regions:
  spiky_data[plateau_region[0]:plateau_region[1]] = 1

plt.plot(spiky_data)
plt.title('this is a spiky function that we want to smooth')
plt.show()

# next build a double-cosine smoothing kernel
width = 10

kernel = np.zeros(N)
kernel[:width] = 0.5 + 0.5*np.cos(np.linspace(0, np.pi, width))
kernel[-width:] = 0.5 + 0.5*np.cos(np.linspace(np.pi, 0, width))
kernel /= np.sum(kernel)

plt.plot(kernel)
plt.title('we want to smooth every point to have this cosine-like taper')
plt.show()

# perform the convolution brute-force
smoothed_data = np.zeros(N)
for n in range(N):
  for m in range(N):
    smoothed_data[n] += kernel[m] * spiky_data[n - m]
plt.plot(smoothed_data)
plt.title('this is the smoothed function')
plt.show()

We see that the convolution does perform the "smoothing" we expect, but we needed to do an ugly, two-level python for loop to implement it. Fortunately, we can do something prettier (and faster) with the DFTs, thanks to the [convolution theorem](https://en.wikipedia.org/wiki/Convolution_theorem). Said simply:
$$ DFT(f*g) = DFT(f)DFT(g)$$
or
$$(f*g) = iDFT(DFT(f)DFT(g))$$
Let's try it out!

In [ ]:
# fft of spiky function
fft_of_spiky_data = np.fft.fft(spiky_data)

# fft of smoothing kernel
fft_of_kernel = np.fft.fft(kernel)

# multiply
fft_of_smoothed_data = fft_of_spiky_data * fft_of_kernel

# inverse fft
smoothed_data = np.fft.ifft(fft_of_smoothed_data).real

plt.plot(smoothed_data)
plt.title('this is the smoothed function')
plt.show()

Look familiar?

Why is this better? Doing it brute-force involved $N^2$ steps. The DFT is very fast, being computed in $~Nlog(N)$ time. So 2 DFTs, 1 inverse DFT, and the multiplication adds up to $N(3log(N) + 1)$ complexity. For large $N$, $3log(N) + 1$ will be much less than $N$, so the DFT approach is much faster.

Earlier, we mentioned how the fact that the DFT and inverse DFT could be swapped has an important implication for convolution. It has to do with the fact that we defined the smoothing kernel *symmetrically around zero* above. What would have happened if we defined the smoothing kernel starting from 0?

In [ ]:
# build a double-cosine smoothing kernel that *starts* at 0 rather
# than being *centered* on 0
width = 10

new_kernel = np.zeros(N)
new_kernel[:2*width] = 0.5 + 0.5*np.cos(np.linspace(-np.pi, np.pi, 2*width))
new_kernel /= np.sum(new_kernel)

plt.plot(new_kernel)
plt.title('we want to smooth every point to have this cosine-like taper')
plt.show()

# get the fft of it and do the convolution
fft_of_new_kernel = np.fft.fft(new_kernel)

# multiply
new_fft_of_smoothed_data = fft_of_spiky_data * fft_of_new_kernel

# inverse fft
new_smoothed_data = np.fft.ifft(new_fft_of_smoothed_data).real

plt.plot(new_smoothed_data)
for x in spiky_points:
  plt.axvline(x, linestyle='--', color='k', alpha=0.5)
for x in plateau_regions:
  plt.axvspan(x[0], x[1], alpha=0.25, color='k')
plt.title('this is the smoothed function')
plt.show()

In grey, we have overplotted the location of the hard edges in the original spiky data. The smoothed data has been shifted to the right by 10!

Remember how the Fourier frequencies were centered at 0, and they went up for positive array indexes and down for negative array indexes? Well, real space *is* the "Fourier space of Fourier space". The DFT and inverse-DFT routines also expect the "center" of real space to be at the 0'th index! We *always* need to respect the numpy FFT conventions when using numpy to perform the transforms! The fact that "real space" is the starting point is simply an illusion: to the code, "real space" and "Fourier space" are not distinguishable other than they are related by the DFT and inverse DFT.

The course will perform many convolutions. Sometimes, the DFT of the kernel is provided for us --- that makes our life easier!

# The 2-dimensional DFT

Now that you've mastered the 1-dimensional Fourier transform and all its subtleties, moving to 2-dimensions is a much smaller hill to climb. That's because the 2-d DFT simply generalizes to something like the outer product of two 1-d DFTs. Again, following the [numpy convention](https://numpy.org/doc/stable/reference/routines.fft.html#higher-dimensions):

![2d_DFT.png](https://raw.githubusercontent.com/flatironinstitute/2026-flatiron-cmb-summer-school/main/notebooks/2d_DFT.png)

Here, $a_mn$ is a 2-d array holding the 2-d signal (for example, an image), and now we tack on the extra $l$ frequency index which behaves just like $k$ in the 1-d case (but is independent from it).

As before, *each axis* follows the centered-at-0 frequency index convention while being constant over the other axis:

In [ ]:
# we can think of m as being the y-axis, k as being the frequency in the y direction (k_y),
# n as being the x-axis, and l as being the frequency in the x direction (k_x).

# so if we have a 2d array of frequency components A_ky_kx coming from a signal
# of shape (N_y, N_x), then the 2-d arrays of frequencies would look like:
N_y = 150
N_x = 200

k_y = np.broadcast_to(np.fft.fftfreq(N_y)[:, None], (N_y, N_x))
k_x = np.broadcast_to(np.fft.fftfreq(N_x), (N_y, N_x))

plt.imshow(k_y, origin='lower')
plt.colorbar()
plt.title('k_y')
plt.show()

plt.imshow(k_x, origin='lower')
plt.colorbar()
plt.title('k_x')
plt.show()


We only really need to be careful with this when building a convolution kernel, since it means (for example, if we'd like to smooth an image using Fourier transforms) that the smoothing shape needs to be centered "in the corners".

Let's try this out. First, let's make a smiley face with hard edges...

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thanks Gemini for this...

# Image dimensions
size = 200
image = np.zeros((size, size), dtype=int)

# Head: A large circle
center_x, center_y = size // 2, size // 2
radius_head = size // 2 - 10

for y in range(size):
    for x in range(size):
        if (x - center_x)**2 + (y - center_y)**2 < radius_head**2:
            image[y, x] = 1

# Eyes: Two smaller circles
radius_eye = size // 10

# Left eye
eye_left_x, eye_left_y = center_x - size // 5, center_y - size // 5
for y in range(size):
    for x in range(size):
        if (x - eye_left_x)**2 + (y - eye_left_y)**2 < radius_eye**2:
            image[y, x] = 0 # 'erase' a part of the head for the eye

# Right eye
eye_right_x, eye_right_y = center_x + size // 5, center_y - size // 5
for y in range(size):
    for x in range(size):
        if (x - eye_right_x)**2 + (y - eye_right_y)**2 < radius_eye**2:
            image[y, x] = 0 # 'erase' a part of the head for the eye

# Mouth: A semi-circle (curved line)
radius_mouth = size // 4
mouth_center_x, mouth_center_y = center_x, center_y + size // 8

for y in range(size):
    for x in range(size):
        dist_sq = (x - mouth_center_x)**2 + (y - mouth_center_y)**2
        # Create an arc for the mouth. The inner and outer radii define the thickness.
        # This makes a 'slice' of a circle, then only takes the bottom half.
        if radius_mouth**2 < dist_sq < (radius_mouth + size // 20)**2 and y > mouth_center_y:
            image[y, x] = 0 # 'erase' a part of the head for the mouth

# Display the image
plt.figure(figsize=(6, 6))
plt.imshow(image, origin='upper')
plt.title('200x200 Hard-edge Smiley Face')
plt.show()

That's mildly creepy...we'll make it a little more friendly by smoothing all the edges.

Let's build a smoothing kernel that is radially symmetric:

In [ ]:
# let's "hijack" np.fft.fftfreq to get the position arrays in real
# space y and x. we know for array of size N, the min and max of
# np.fft.fftfreq is +/- 0.5. So just multiply by N:
y = np.fft.fftfreq(size) * size
x = np.fft.fftfreq(size) * size
r = np.sqrt(x[:, None]**2 + y[None, :]**2)

plt.imshow(r)
plt.colorbar()
plt.title('r')
plt.show()

# now get a width of 10 pixels
width = 10
kernel = 0.5 + 0.5*np.cos(np.pi * np.minimum(r, width) / width)
kernel /= np.sum(kernel)

plt.imshow(kernel)
plt.colorbar()
plt.title('kernel')
plt.show()

Can you do the rest? Smooth the face with the kernel please! The 2-d DFT and its inverse are `np.fft.fft2` and `np.fft.ifft2`:

In [ ]:
# your code here

Not sure that's any less creepy...